# 5CM522 May 2026 In-Class Test Worked Solutions

This notebook runs locally in the project environment or in Google Colab.

See the setup note below before running.

**Setup.** Run this notebook in the project's conda environment (see the README at the top of the repository), which already contains Pyomo, HiGHS, SimPy and OR-Tools.

If you are using Google Colab instead, first run this in a new cell:

```
!pip install pyomo simpy ortools openpyxl highspy
```

In [ ]:
import random
import numpy as np
import pandas as pd
import simpy
try:
    from IPython.display import display as show
except ModuleNotFoundError:
    def show(x):
        print(x)
from ortools.sat.python import cp_model
from pyomo.environ import *

pd.set_option("display.max_columns", None)

def get_solver():
    for name in ("appsi_highs", "highs"):
        solver = SolverFactory(name)
        if solver.available(False):
            return solver
    raise RuntimeError("No suitable Pyomo solver found. In Colab, make sure highspy installed correctly.")

solver = get_solver()
print("Solver ready:", solver)

## Question 1

In [ ]:
def build_q1(domain, printing_hours=84, finishing_hours=60, premium_share=False):
    model = ConcreteModel()
    model.x = Var(domain=domain)
    model.y = Var(domain=domain)
    model.obj = Objective(expr=42 * model.x + 36 * model.y, sense=maximize)
    model.printing = Constraint(expr=4 * model.x + 2 * model.y <= printing_hours)
    model.finishing = Constraint(expr=2 * model.x + 3 * model.y <= finishing_hours)
    model.market = Constraint(expr=model.y <= 18)
    if premium_share:
        model.share = Constraint(expr=model.y >= 0.30 * (model.x + model.y))
    solver.solve(model)
    return model

def summarise_q1(domain):
    baseline = build_q1(domain)
    extra_printing = build_q1(domain, printing_hours=90)
    extra_finishing = build_q1(domain, finishing_hours=66)
    premium_share = build_q1(domain, premium_share=True)

    x = value(baseline.x)
    y = value(baseline.y)
    z = value(baseline.obj)

    return {
        "x": round(x, 4),
        "y": round(y, 4),
        "contribution": round(z, 4),
        "printing_slack": round(84 - (4 * x + 2 * y), 4),
        "finishing_slack": round(60 - (2 * x + 3 * y), 4),
        "market_slack": round(18 - y, 4),
        "with_6_extra_printing": round(value(extra_printing.obj), 4),
        "with_6_extra_finishing": round(value(extra_finishing.obj), 4),
        "premium_share_solution": (round(value(premium_share.x), 4), round(value(premium_share.y), 4)),
    }

print("Continuous solution")
print(summarise_q1(NonNegativeReals))
print()
print("Integer solution")
print(summarise_q1(NonNegativeIntegers))

## Question 2

In [ ]:
sites = ["A", "B", "C", "D", "E", "F"]
districts = [1, 2, 3, 4, 5, 6]
costs = {"A": 16, "B": 11, "C": 15, "D": 14, "E": 13, "F": 10}
coverage = {
    "A": [1, 2, 4],
    "B": [1, 3],
    "C": [2, 3, 5],
    "D": [4, 5, 6],
    "E": [2, 5, 6],
    "F": [3, 4, 6],
}

def build_q2(backup_for_6=False, forbid_bf=False):
    model = ConcreteModel()
    model.S = Set(initialize=sites)
    model.D = Set(initialize=districts)
    model.x = Var(model.S, domain=Binary)
    model.obj = Objective(expr=sum(costs[s] * model.x[s] for s in model.S), sense=minimize)
    model.cover = Constraint(model.D, rule=lambda m, d: sum(m.x[s] for s in m.S if d in coverage[s]) >= 1)
    if backup_for_6:
        model.backup = Constraint(expr=sum(model.x[s] for s in model.S if 6 in coverage[s]) >= 2)
    if forbid_bf:
        model.no_bf = Constraint(expr=model.x["B"] + model.x["F"] <= 1)
    solver.solve(model)
    return model

baseline = build_q2()
revised = build_q2(backup_for_6=True, forbid_bf=True)

baseline_sites = [s for s in sites if value(baseline.x[s]) > 0.5]
revised_sites = [s for s in sites if value(revised.x[s]) > 0.5]

print("Baseline sites:", baseline_sites)
print("Baseline cost:", value(baseline.obj))
print()
print("Revised sites:", revised_sites)
print("Revised cost:", value(revised.obj))

## Question 3

In [ ]:
depots = ["P1", "P2", "P3"]
hubs = ["H1", "H2"]
clinics = ["C1", "C2", "C3", "C4"]

supply = {"P1": 100, "P2": 120, "P3": 80}
demand = {"C1": 70, "C2": 60, "C3": 80, "C4": 50}

cost_in = {
    ("P1", "H1"): 4, ("P1", "H2"): 6,
    ("P2", "H1"): 5, ("P2", "H2"): 3,
    ("P3", "H1"): 6, ("P3", "H2"): 4,
}

cost_out = {
    ("H1", "C1"): 4, ("H1", "C2"): 6, ("H1", "C3"): 7, ("H1", "C4"): 5,
    ("H2", "C1"): 5, ("H2", "C2"): 4, ("H2", "C3"): 3, ("H2", "C4"): 6,
}

def build_q3(h1_cap=None, h1_c2_min=None, direct_route=False):
    model = ConcreteModel()
    model.P = Set(initialize=depots)
    model.H = Set(initialize=hubs)
    model.C = Set(initialize=clinics)
    model.inflow = Var(model.P, model.H, domain=NonNegativeReals)
    model.outflow = Var(model.H, model.C, domain=NonNegativeReals)

    total_cost = sum(cost_in[p, h] * model.inflow[p, h] for p in model.P for h in model.H)
    total_cost += sum(cost_out[h, c] * model.outflow[h, c] for h in model.H for c in model.C)

    if direct_route:
        model.direct = Var(domain=NonNegativeReals, bounds=(0, 20))
        total_cost += 9 * model.direct

    model.obj = Objective(expr=total_cost, sense=minimize)
    model.supply = Constraint(model.P, rule=lambda m, p: sum(m.inflow[p, h] for h in m.H) <= supply[p])
    model.demand = Constraint(
        model.C,
        rule=lambda m, c: sum(m.outflow[h, c] for h in m.H) + (m.direct if direct_route and c == "C4" else 0) == demand[c],
    )
    model.balance = Constraint(model.H, rule=lambda m, h: sum(m.inflow[p, h] for p in m.P) == sum(m.outflow[h, c] for c in m.C))

    if h1_cap is not None:
        model.h1_cap = Constraint(expr=sum(model.outflow["H1", c] for c in model.C) <= h1_cap)
    if h1_c2_min is not None:
        model.h1_c2_min = Constraint(expr=model.outflow["H1", "C2"] >= h1_c2_min)
    if direct_route:
        model.direct_supply = Constraint(expr=sum(model.inflow["P3", h] for h in model.H) + model.direct <= supply["P3"])

    solver.solve(model)
    return model

def non_zero_flows(model):
    rows = []
    for p in model.P:
        for h in model.H:
            if value(model.inflow[p, h]) > 1e-6:
                rows.append(("Depot to Hub", p, h, round(value(model.inflow[p, h]), 4)))
    for h in model.H:
        for c in model.C:
            if value(model.outflow[h, c]) > 1e-6:
                rows.append(("Hub to Clinic", h, c, round(value(model.outflow[h, c]), 4)))
    return rows

baseline = build_q3()
revised = build_q3(h1_cap=130, h1_c2_min=30)
emergency = build_q3(h1_cap=130, h1_c2_min=30, direct_route=True)

print("Baseline cost:", value(baseline.obj))
print(non_zero_flows(baseline))
print()
print("Revised cost:", value(revised.obj))
print(non_zero_flows(revised))
print()
print("Emergency-route cost:", value(emergency.obj))
print(non_zero_flows(emergency))
print("Direct P3 to C4:", value(emergency.direct))

## Question 4

In [ ]:
def run_helpdesk(arrival_rate_per_hour, mean_service_minutes, technicians, run_minutes=420, seed=42, target_wait=10):
    rng = random.Random(seed)
    waits = []

    def student(env, desk):
        arrival = env.now
        with desk.request() as request:
            yield request
            waits.append(env.now - arrival)
            service = rng.expovariate(1 / mean_service_minutes)
            yield env.timeout(service)

    def arrivals(env, desk):
        while True:
            interarrival = rng.expovariate(arrival_rate_per_hour / 60)
            yield env.timeout(interarrival)
            if env.now > run_minutes:
                break
            env.process(student(env, desk))

    env = simpy.Environment()
    desk = simpy.Resource(env, capacity=technicians)
    env.process(arrivals(env, desk))
    env.run(until=run_minutes + 10000)

    return {
        "served": len(waits),
        "avg_wait": round(sum(waits) / len(waits), 4),
        "max_wait": round(max(waits), 4),
        "pct_under_10": round(100 * sum(w < target_wait for w in waits) / len(waits), 4),
    }

def run_mixed_helpdesk(arrival_rate_per_hour, technicians, run_minutes=420, seed=42, target_wait=10):
    rng = random.Random(seed)
    waits = []
    complex_cases = 0

    def student(env, desk, is_complex):
        nonlocal complex_cases
        arrival = env.now
        with desk.request() as request:
            yield request
            waits.append(env.now - arrival)
            if is_complex:
                complex_cases += 1
            mean_service = 14 if is_complex else 6
            service = rng.expovariate(1 / mean_service)
            yield env.timeout(service)

    def arrivals(env, desk):
        while True:
            interarrival = rng.expovariate(arrival_rate_per_hour / 60)
            yield env.timeout(interarrival)
            if env.now > run_minutes:
                break
            env.process(student(env, desk, rng.random() < 0.25))

    env = simpy.Environment()
    desk = simpy.Resource(env, capacity=technicians)
    env.process(arrivals(env, desk))
    env.run(until=run_minutes + 10000)

    return {
        "served": len(waits),
        "avg_wait": round(sum(waits) / len(waits), 4),
        "max_wait": round(max(waits), 4),
        "pct_under_10": round(100 * sum(w < target_wait for w in waits) / len(waits), 4),
        "complex_cases": complex_cases,
    }

baseline = run_helpdesk(9, 8, 2)
busy_2 = run_helpdesk(13, 8, 2)
busy_3 = run_helpdesk(13, 8, 3)
busy_4 = run_helpdesk(13, 8, 4)
mixed = run_mixed_helpdesk(13, 3)

print("Baseline with 2 technicians:")
print(baseline)
print()
print("Busy period with 2 technicians:")
print(busy_2)
print()
print("Busy period with 3 technicians:")
print(busy_3)
print()
print("Busy period with 4 technicians:")
print(busy_4)
print()
print("Mixed service with 3 technicians:")
print(mixed)

## Question 5

In [ ]:
data = [380, 505, 455, 590, 410, 550, 500, 645, 445, 600, 540, 705, 490, 660, 590, 775]
periods = np.arange(1, len(data) + 1)
quarters = ((periods - 1) % 4) + 1

df = pd.DataFrame({"t": periods, "quarter": quarters, "actual": data})
df["ma4"] = pd.Series(data).rolling(4).mean()
df["cma"] = (df["ma4"] + df["ma4"].shift(-1)) / 2

train = df[df["t"] <= 12].copy()
seasonal = train.dropna().copy()
seasonal["add_seasonal"] = seasonal["actual"] - seasonal["cma"]
seasonal["mult_seasonal"] = seasonal["actual"] / seasonal["cma"]

add_indices = seasonal.groupby("quarter")["add_seasonal"].mean()
add_indices = add_indices - add_indices.mean()

mult_indices = seasonal.groupby("quarter")["mult_seasonal"].mean()
mult_indices = mult_indices / mult_indices.mean()

train["deseasonalised_add"] = train["actual"] - train["quarter"].map(add_indices)
train["deseasonalised_mult"] = train["actual"] / train["quarter"].map(mult_indices)

add_slope, add_intercept = np.polyfit(train["t"], train["deseasonalised_add"], 1)
mult_slope, mult_intercept = np.polyfit(train["t"], train["deseasonalised_mult"], 1)

validation = df[df["t"] >= 13].copy()
validation["add_forecast"] = add_slope * validation["t"] + add_intercept + validation["quarter"].map(add_indices)
validation["mult_forecast"] = (mult_slope * validation["t"] + mult_intercept) * validation["quarter"].map(mult_indices)
validation["add_sq_error"] = (validation["actual"] - validation["add_forecast"]) ** 2
validation["mult_sq_error"] = (validation["actual"] - validation["mult_forecast"]) ** 2

future = pd.DataFrame({"t": [17, 18, 19, 20]})
future["quarter"] = ((future["t"] - 1) % 4) + 1
future["add_forecast"] = add_slope * future["t"] + add_intercept + future["quarter"].map(add_indices)
future["mult_forecast"] = (mult_slope * future["t"] + mult_intercept) * future["quarter"].map(mult_indices)

mse_add = validation["add_sq_error"].mean()
mse_mult = validation["mult_sq_error"].mean()

print("Data with moving averages and centred moving averages")
show(df)
print()
print("Additive seasonal indices")
print(add_indices.round(4).to_dict())
print()
print("Multiplicative seasonal indices")
print(mult_indices.round(4).to_dict())
print()
print("Validation results")
show(validation[["t", "actual", "add_forecast", "mult_forecast", "add_sq_error", "mult_sq_error"]].round(4))
print("MSE additive:", round(mse_add, 4))
print("MSE multiplicative:", round(mse_mult, 4))
print("Better model:", "additive" if mse_add <= mse_mult else "multiplicative")
print()
print("2023 forecasts")
show(future.round(4))

# Optional workbook export:
# with pd.ExcelWriter("in_class_test_forecasting.xlsx", engine="openpyxl") as writer:
#     df.to_excel(writer, sheet_name="Data and CMA", index=False)
#     validation.to_excel(writer, sheet_name="Validation", index=False)
#     future.to_excel(writer, sheet_name="Future Forecasts", index=False)

## Question 6

In [ ]:
flow_jobs = {"J1": (8, 4), "J2": (3, 9), "J3": (6, 5), "J4": (7, 2), "J5": (4, 6)}

remaining = dict(flow_jobs)
front = []
back = []
while remaining:
    next_job = min(remaining, key=lambda job: min(remaining[job]))
    m1, m2 = remaining.pop(next_job)
    if m1 <= m2:
        front.append(next_job)
    else:
        back.insert(0, next_job)
johnson_order = front + back

def flow_schedule(order):
    m1_clock = 0
    m2_clock = 0
    rows = []
    for job in order:
        m1_time, m2_time = flow_jobs[job]
        m1_start = m1_clock
        m1_end = m1_start + m1_time
        m1_clock = m1_end
        m2_start = max(m2_clock, m1_end)
        m2_end = m2_start + m2_time
        m2_clock = m2_end
        rows.append([job, m1_start, m1_end, m2_start, m2_end])
    return m2_clock, pd.DataFrame(rows, columns=["job", "m1_start", "m1_end", "m2_start", "m2_end"])

johnson_makespan, johnson_table = flow_schedule(johnson_order)
original_makespan, original_table = flow_schedule(["J1", "J2", "J3", "J4", "J5"])

print("Johnson order:", johnson_order)
print("Johnson makespan:", johnson_makespan)
show(johnson_table)
print("Original-order makespan:", original_makespan)
show(original_table)

job_shop_data = {
    "A": [("M1", 3), ("M3", 5), ("M2", 2)],
    "B": [("M2", 4), ("M1", 2), ("M3", 6)],
    "C": [("M3", 3), ("M2", 4), ("M1", 3)],
    "D": [("M1", 2), ("M2", 6), ("M3", 4)],
}

def solve_job_shop(job_data, due_dates=None, fixed_starts=None):
    model = cp_model.CpModel()
    machine_names = sorted({machine for tasks in job_data.values() for machine, _ in tasks})
    horizon = sum(duration for tasks in job_data.values() for _, duration in tasks)

    tasks = {}
    machine_intervals = {machine: [] for machine in machine_names}

    for job_name, job_tasks in job_data.items():
        for i, (machine_name, duration) in enumerate(job_tasks):
            start = model.NewIntVar(0, horizon, f"start_{job_name}_{i}")
            end = model.NewIntVar(0, horizon, f"end_{job_name}_{i}")
            interval = model.NewIntervalVar(start, duration, end, f"interval_{job_name}_{i}")
            tasks[(job_name, i)] = {"machine": machine_name, "duration": duration, "start": start, "end": end}
            machine_intervals[machine_name].append(interval)

    if fixed_starts:
        for task_key, start_time in fixed_starts.items():
            model.Add(tasks[task_key]["start"] == start_time)

    for intervals in machine_intervals.values():
        model.AddNoOverlap(intervals)

    for job_name, job_tasks in job_data.items():
        for i in range(len(job_tasks) - 1):
            model.Add(tasks[(job_name, i + 1)]["start"] >= tasks[(job_name, i)]["end"])

    if due_dates:
        for job_name, due_date in due_dates.items():
            last = len(job_data[job_name]) - 1
            model.Add(tasks[(job_name, last)]["end"] <= due_date)

    makespan = model.NewIntVar(0, horizon, "makespan")
    model.AddMaxEquality(makespan, [tasks[(job_name, len(job_tasks) - 1)]["end"] for job_name, job_tasks in job_data.items()])
    model.Minimize(makespan)

    cp_solver = cp_model.CpSolver()
    cp_solver.parameters.max_time_in_seconds = 10
    cp_solver.parameters.num_search_workers = 1
    cp_solver.parameters.random_seed = 42
    status = cp_solver.Solve(model)

    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return cp_solver.StatusName(status), None, pd.DataFrame(columns=["job", "task", "machine", "duration", "start", "end"])

    rows = []
    for job_name, job_tasks in job_data.items():
        for i, (machine_name, duration) in enumerate(job_tasks):
            rows.append([
                job_name,
                i + 1,
                machine_name,
                duration,
                cp_solver.Value(tasks[(job_name, i)]["start"]),
                cp_solver.Value(tasks[(job_name, i)]["end"]),
            ])

    schedule = pd.DataFrame(rows, columns=["job", "task", "machine", "duration", "start", "end"])
    schedule = schedule.sort_values(["start", "job", "task"]).reset_index(drop=True)
    return cp_solver.StatusName(status), cp_solver.Value(makespan), schedule

status, makespan, schedule = solve_job_shop(job_shop_data)
print()
print("Baseline job-shop status:", status)
print("Baseline job-shop makespan:", makespan)
show(schedule)

revised_job_shop_data = dict(job_shop_data)
revised_job_shop_data["BLOCK"] = [("M2", 3)]
revised_status, revised_makespan, revised_schedule = solve_job_shop(
    revised_job_shop_data,
    due_dates={"D": 15},
    fixed_starts={("BLOCK", 0): 6},
)
print("Revised job-shop status:", revised_status)
print("Revised job-shop makespan:", revised_makespan)
show(revised_schedule)